# Phase 6: Evidence Reconciliation, Exceptions, and Conditions

Specialists can describe the same loan in different ways. This phase creates canonical facts, deduplicates exception codes, links findings to rule IDs, and proposes conditions for human review.

In [ ]:
# Locate the src-layout project and import the completed earlier phases.
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / "underwritingAgent", Path.cwd().parent / "underwritingAgent"]
PROJECT_ROOT = next(path.resolve() for path in candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PDF_ROOT = PROJECT_ROOT / "data" / "realistic_pdfs"
GUIDELINES = PROJECT_ROOT / "data" / "underwriting_guidelines.jsonl"
INTAKE_REFERENCES = ["UW-26-0417-A", "BRK-90831", "WHL-77-2206"]
print(PROJECT_ROOT)


In [ ]:
from underwriting_agent.document_layer import build_document_workflow
from underwriting_agent.borrower_analysis import build_borrower_workflow
from underwriting_agent.property_analysis import build_property_workflow
from underwriting_agent.property_research import build_property_research_workflow
from underwriting_agent.calculations_policy import build_calculation_policy_workflow
from underwriting_agent.intake_packages import resolve_document_paths

def through_phase_5(loan_id):
    paths = resolve_document_paths(PDF_ROOT, loan_id)
    state = build_document_workflow().invoke({"loan_id": loan_id, "document_paths": [str(path) for path in paths], "workflow_status": "INTAKE"})
    state = build_borrower_workflow().invoke(state)
    state = build_property_workflow().invoke(state)
    state = build_property_research_workflow().invoke(state)
    return build_calculation_policy_workflow(GUIDELINES).invoke(state)


## 1. Reconcile specialist outputs into canonical facts

Each important fact carries source document IDs. Calculated facts have no source document because they are derived deterministically from other canonical facts.

In [ ]:
from underwriting_agent.reconciliation import reconcile_facts_node

phase5_state = through_phase_5("WHL-77-2206")
facts_update = reconcile_facts_node(phase5_state)
[fact.model_dump(mode="json") for fact in facts_update["canonical_facts"]]


## 2. Normalize exceptions and create conditions

In [ ]:
from underwriting_agent.reconciliation import build_reconciliation_workflow

phase6 = build_reconciliation_workflow()
result = phase6.invoke(phase5_state)
print("Status:", result["workflow_status"])
print("Exceptions:")
for item in result["exceptions"]:
    print(" -", item.code, item.severity, item.rule_ids)
print("Conditions:")
for condition in result["conditions"]:
    print(" -", condition)


## 3. Compare clean, missing-document, and showcase cases

In [ ]:
comparison = []
for loan_id in INTAKE_REFERENCES:
    result = phase6.invoke(through_phase_5(loan_id))
    comparison.append({"loan_id": loan_id, "status": result["workflow_status"], "exceptions": [item.code for item in result["exceptions"]], "conditions": result["conditions"]})
comparison


## Phase 6 handoff

Phase 7 receives canonical facts, retrieved rules, normalized exceptions, and proposed conditions—all structured and traceable.

## Optional production services: OpenAI and Pinecone

This phase intentionally remains deterministic. An OpenAI model may normalize messy source documents in Phase 2 and draft reviewer-facing prose in Phase 7, while borrower routing, income selection, asset exclusion, appraisal comparison, reconciliation, and exception rules remain typed Python logic. Pinecone is used only for guideline retrieval in Phase 5.